# M3 · Graph ile Anlamsallaştırma ve Kişiselleştirme (GraphRAG)
**Süre:** 20 dk canlı kodlama · **Demo çıktısı:** Kullanıcı profiline göre sıralanan sonuçlar

**Vektör araması ne bilmez?**
- Kavramlar arasındaki **ilişkileri** ("avans → tedarikçi → risk")
- **Kim** soruyor? Finans müdürü ile güvenlik uzmanı için "önemli" olan farklıdır.

```
  (Ayşe)──ILGILENIR──▶(avans)◀──ILGILI──(DOK-03 Avans Politikası)
                         │
                      BAĞLIDIR
                         ▼
                    (tedarikçi)◀──ILGILI──(DOK-02 Tedarikçi Prosedürü)
```
**Fikir:** Skor = α · vektör benzerliği + (1-α) · grafta kullanıcıya yakınlık (Kişiselleştirilmiş PageRank)

In [ ]:
# ⚙️ Ortam hazırlığı: Colab'da ve yerel Jupyter'da çalışır
import os, sys, subprocess
REPO_URL = "https://github.com/volkanak/MBT-GenAI-Egitim.git"   # ← kendi GitHub reponuz
KOK = next((os.path.abspath(p) for p in [".", "..", "/content/MBT-GenAI-Egitim"]
            if os.path.exists(os.path.join(p, "ortak.py"))), None)
if KOK is None:  # Colab: repoyu klonla
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/MBT-GenAI-Egitim"], check=True)
    KOK = "/content/MBT-GenAI-Egitim"
try:
    import langgraph, chromadb, langchain_anthropic, sentence_transformers, langchain_mcp_adapters  # noqa
except ImportError:
    print("Paketler kuruluyor (1-3 dk)…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KOK}/requirements.txt"], check=True)
sys.path.insert(0, KOK); os.chdir(KOK)
from ortak import *
api_anahtari_yukle()
print("Hazır ✔  Model:", MODEL)

## 1️⃣ LLM ile dokümandan bilgi (üçlü) çıkarma  ✍️ *canlı kodlama*
GraphRAG'in ilk adımı: metinden **(varlık, ilişki, varlık)** üçlüleri çıkarmak. Canlıda 1 doküman çalıştırıyoruz; tamamı `data/graf_ucluleri.json` içinde hazır.

In [ ]:
from pydantic import BaseModel, Field

# ✍️ TODO: Uclu ve Ucluler şemalarını tanımlayıp DOK-03'ten üçlü çıkarın
...
for u in cikti.ucluler:
    print(f"({u.kaynak}) -[{u.iliski}]-> ({u.hedef})")

## 2️⃣ Bilgi grafını kur ve çiz
Düğümler: 👤 kullanıcı · 📄 doküman · 🏷️ konu

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = graf_kur()   # ortak.py: dokümanlar + kullanıcılar + konu ilişkileri
print(G.number_of_nodes(), "düğüm,", G.number_of_edges(), "kenar")

renk = {"kullanici": "#e4572e", "dokuman": "#4c78a8", "konu": "#9ecae9"}
plt.figure(figsize=(13, 9))
pos = nx.spring_layout(G, seed=7, k=0.6)
nx.draw_networkx(G, pos, node_color=[renk[G.nodes[n]["tur"]] for n in G],
                 labels={n: G.nodes[n]["etiket"][:22] for n in G}, font_size=7,
                 node_size=[900 if G.nodes[n]["tur"] != "konu" else 400 for n in G], edge_color="#ccc")
plt.axis("off"); plt.show()

## 3️⃣ Kişiselleştirilmiş PageRank: grafta "bana yakın" olan ne?  ✍️ *canlı kodlama*
Rastgele yürüyüş her seferinde **kullanıcının düğümüne geri döner** → kullanıcıya yakın dokümanlar yüksek skor alır.

In [ ]:
kullanicilar = kullanicilari_yukle()

# ✍️ TODO: Her kullanıcı için personalization={kullanici_id: 1} ile PageRank hesaplayıp ilk 4 dokümanı yazdırın
...

## 4️⃣ Vektör + graf = kişiselleştirilmiş sıralama  ✍️ *canlı kodlama*

In [ ]:
# ✍️ TODO: Geniş aday havuzu al, doküman başına en iyi parçayı tut, final = alfa*vektör + (1-alfa)*graf
...

## 5️⃣ Aynı soru, üç farklı kullanıcı

In [ ]:
import pandas as pd

SORU = "Bu çeyrekte dikkat etmem gereken kurallar neler?"
tablo = {}
for u in kullanicilar:
    tablo[f"{u['ad']} · {u['unvan']}"] = [f"{r['baslik'][:34]} ({r['final']})" for r in kisisel_ara(SORU, u["id"])[:4]]
tablo["Sadece vektör (kişiselleştirme yok)"] = [f"{r['baslik'][:34]} ({r['skor']})" for r in rag_indeksi().ara(SORU, 4)]
pd.DataFrame(tablo, index=[1, 2, 3, 4])

## 6️⃣ Kişiselleştirilmiş cevap

In [ ]:
def kisisel_cevap(soru: str, kullanici: dict) -> str:
    sonuclar = kisisel_ara(soru, kullanici["id"])[:4]
    sistem = (RAG_SISTEM + f"\nKullanıcı: {kullanici['ad']}, {kullanici['unvan']} ({kullanici['departman']}). "
              "Cevabı bu kişinin rolü ve sorumluluklarına göre önceliklendir, en fazla 5 madde yaz.")
    return metin_sor(f"Bağlam:\n{baglam_olustur(sonuclar)}\n\nSoru: {soru}", sistem=sistem)

for u in kullanicilar[:2]:
    print(f"━━━━ 👤 {u['ad']} ━━━━\n{kisisel_cevap(SORU, u)}\n")

### ✅ Özet ve tartışma
- **alfa** değerini 1.0 yapın → kişiselleştirme kapanır. 0.3 yapın → profil baskın olur. Doğru değer A/B testiyle bulunur.
- Üretimde: Neo4j / Memgraph / Azure Cosmos DB (Gremlin), Microsoft GraphRAG (topluluk özetleri), LightRAG.
- Graf; **yetkilendirme** (kim hangi dokümanı görebilir) için de kullanılabilir.
- Şimdiye kadar **biz** hangi adımın çalışacağına karar verdik → **M4:** model araçları kendisi seçsin (ajan).